In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 45.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 69.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.3/692.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Thi

In [3]:
"""Main function to run the entire pipeline with enhanced feature engineering"""
print("Starting IEEE-CIS Fraud Detection Pipeline with Magic Features")

# Define data paths - update these to your actual file paths
transaction_path = "/kaggle/input/ieee-fraud-detection/train_transaction.csv"  # Update this path
identity_path = "/kaggle/input/ieee-fraud-detection/train_identity.csv"  # Update this path


Starting IEEE-CIS Fraud Detection Pipeline with Magic Features


In [4]:
df = pd.read_csv(transaction_path)
id_df = pd.read_csv(identity_path)

In [5]:
mlflow.set_experiment('DecisionTree_Training')

<Experiment: artifact_location='mlflow-artifacts:/90ac70d8239143c0a00a690e758ad6db', creation_time=1745761114156, experiment_id='6', last_update_time=1745761114156, lifecycle_stage='active', name='DecisionTree_Training', tags={}>

In [9]:
# IEEE-CIS Fraud Detection Pipeline with Magic Features
# Complete implementation with enhanced feature engineering

import numpy as np
import pandas as pd
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

def load_ieee_cis_data(transaction_path, identity_path):
    """
    Load IEEE-CIS Fraud Detection dataset from provided file paths
    
    Parameters:
    - transaction_path: Path to the transaction CSV file
    - identity_path: Path to the identity CSV file
    
    Returns:
    - DataFrame containing merged transaction and identity data
    """
    try:
        transaction_df = pd.read_csv(transaction_path)
        print(f"Transaction data loaded: {transaction_df.shape}")
        
        try:
            identity_df = pd.read_csv(identity_path)
            print(f"Identity data loaded: {identity_df.shape}")
            
            # Merge on TransactionID
            merged_df = transaction_df.merge(identity_df, on='TransactionID', how='left')
            print(f"Merged data shape: {merged_df.shape}")
            
        except FileNotFoundError:
            print(f"Identity file not found at {identity_path}, using only transaction data")
            merged_df = transaction_df
    
    except FileNotFoundError:
        raise FileNotFoundError(f"Transaction file not found at {transaction_path}")
    
    return merged_df

def analyze_and_clean_data(df):
    """
    Analyze and clean the IEEE-CIS dataset
    
    Parameters:
    - df: DataFrame containing transaction and identity data
    
    Returns:
    - Cleaned DataFrame
    """
    mlflow.set_experiment('DecisionTree_Training')
    with mlflow.start_run(run_name="data_cleaning"):
        # Clone DataFrame to avoid modifying the original
        cleaned_df = df.copy()
        
        # Log initial data stats
        mlflow.log_param("initial_shape", str(cleaned_df.shape))
        mlflow.log_param("initial_missing_values", str(cleaned_df.isna().sum().sum()))
        
        # Drop columns with too many missing values (>95%)
        threshold = 0.95 * len(cleaned_df)
        cols_to_drop = [col for col in cleaned_df.columns if cleaned_df[col].isna().sum() > threshold]
        cleaned_df = cleaned_df.drop(cols_to_drop, axis=1)
        
        mlflow.log_param("dropped_cols_count", len(cols_to_drop))
        mlflow.log_param("dropped_cols", str(cols_to_drop))
        
        # Generate data quality report
        columns = list(cleaned_df.columns)
        dtypes = [str(cleaned_df[col].dtype) for col in columns]
        non_null_counts = [cleaned_df[col].count() for col in columns]
        nunique_values = [cleaned_df[col].nunique() for col in columns]
        memory_usage = [cleaned_df[col].memory_usage(deep=True) / 1024**2 for col in columns]
        
        data_quality = pd.DataFrame({
            'column': columns,
            'dtype': dtypes,
            'non_null_count': non_null_counts,
            'nunique': nunique_values,
            'memory_usage': memory_usage
        })
        
        # Save and log data quality report
        data_quality.to_csv("data_quality_report.csv", index=False)
        mlflow.log_artifact("data_quality_report.csv")
        
        # Generate and log class distribution
        if 'isFraud' in cleaned_df.columns:
            fraud_distribution = cleaned_df['isFraud'].value_counts().to_dict()
            mlflow.log_param("class_distribution", str(fraud_distribution))
            
            # Generate and log class distribution plot
            plt.figure(figsize=(8, 5))
            sns.countplot(x='isFraud', data=cleaned_df)
            plt.title('Distribution of Fraud vs Non-Fraud Transactions')
            plt.savefig('class_distribution.png')
            mlflow.log_artifact('class_distribution.png')
            plt.close()
        
        # Log final data stats
        mlflow.log_param("final_shape", str(cleaned_df.shape))
        mlflow.log_param("final_missing_values", str(cleaned_df.isna().sum().sum()))
        
        return cleaned_df

def prepare_feature_lists(df):
    """
    Identify different types of features in the dataset
    
    Parameters:
    - df: DataFrame containing transaction and identity data
    
    Returns:
    - Dictionary containing lists of different feature types
    """
    with mlflow.start_run(run_name="feature_identification"):
        # Drop target variable for feature identification
        if 'isFraud' in df.columns:
            features_df = df.drop('isFraud', axis=1)
        else:
            features_df = df.copy()
        
        # Identify different types of features
        vesta_features = [col for col in features_df.columns if col.startswith('V')]
        id_features = [col for col in features_df.columns if col.startswith('id_')]
        device_features = [col for col in features_df.columns if col.startswith('DeviceInfo') or col.startswith('DeviceType')]
        
        # Numeric features (excluding transaction ID)
        numeric_features = features_df.select_dtypes(include=[np.number]).columns.tolist()
        if 'TransactionID' in numeric_features:
            numeric_features.remove('TransactionID')
        
        # Categorical features
        categorical_features = features_df.select_dtypes(include=['object']).columns.tolist()
        
        # Add M (match) features to categorical if they're not already in numeric
        m_features = [col for col in features_df.columns if col.startswith('M') and col not in numeric_features]
        categorical_features.extend(m_features)
        
        # Log feature counts
        mlflow.log_param("num_vesta_features", len(vesta_features))
        mlflow.log_param("num_id_features", len(id_features))
        mlflow.log_param("num_device_features", len(device_features))
        mlflow.log_param("num_numeric_features", len(numeric_features))
        mlflow.log_param("num_categorical_features", len(categorical_features))
        
        # Save feature lists
        feature_lists = {
            'vesta_features': vesta_features,
            'id_features': id_features,
            'device_features': device_features,
            'numeric_features': numeric_features,
            'categorical_features': categorical_features
        }
        
        # Save feature lists to file and log to MLflow
        with open("feature_lists.txt", "w") as f:
            for feature_type, features in feature_lists.items():
                f.write(f"{feature_type}: {', '.join(features)}\n\n")
        
        mlflow.log_artifact("feature_lists.txt")
        
        return feature_lists


class EnhancedFeatureEngineering(BaseEstimator, TransformerMixin):
    """Enhanced feature engineering for IEEE-CIS dataset with Magic UID feature"""
    
    def __init__(self):
        self.vesta_cols = None
        self.card_cols = None
        # Store the means and stds from training data for each aggregation
        self.agg_stats = {}
    
    def fit(self, X, y=None):
        # Identify columns for grouping
        self.vesta_cols = [col for col in X.columns if col.startswith('V')]
        self.card_cols = [col for col in X.columns if col.startswith('card')]
        
        # Calculate day from TransactionDT
        if 'TransactionDT' in X.columns:
            X_temp = X.copy()
            X_temp['day'] = X_temp['TransactionDT'] / (24*60*60)
            
            # Create UID feature if necessary columns exist
            if 'card1' in X_temp.columns and 'addr1' in X_temp.columns and 'D1' in X_temp.columns:
                # Create card1_addr1 combination
                X_temp['card1_addr1'] = X_temp['card1'].astype(str) + '_' + X_temp['addr1'].astype(str)
                
                # Create UID feature
                X_temp['uid'] = X_temp['card1_addr1'].astype(str) + '_' + (np.floor(X_temp['day'] - X_temp['D1'])).astype(str)
                
                # Store aggregation statistics for numerical features based on UID
                agg_features = []
                
                # Transaction amount and D features
                if all(col in X_temp.columns for col in ['TransactionAmt', 'D4', 'D9', 'D10', 'D15']):
                    for col in ['TransactionAmt', 'D4', 'D9', 'D10', 'D15']:
                        # Make sure the column is numeric before aggregating
                        if pd.api.types.is_numeric_dtype(X_temp[col]):
                            for stat in ['mean', 'std']:
                                agg_col = f'{col}_{stat}_by_uid'
                                agg_result = X_temp.groupby('uid')[col].agg(stat).reset_index()
                                agg_result.columns = ['uid', agg_col]
                                self.agg_stats[agg_col] = agg_result
                                agg_features.append(agg_col)
                
                # C features (except C3)
                c_cols = [f'C{x}' for x in range(1, 15) if x != 3 and f'C{x}' in X_temp.columns]
                for col in c_cols:
                    # Make sure the column is numeric before aggregating
                    if pd.api.types.is_numeric_dtype(X_temp[col]):
                        agg_col = f'{col}_mean_by_uid'
                        agg_result = X_temp.groupby('uid')[col].mean().reset_index()
                        agg_result.columns = ['uid', agg_col]
                        self.agg_stats[agg_col] = agg_result
                        agg_features.append(agg_col)
                
                # M features
                m_cols = [f'M{x}' for x in range(1, 10) if f'M{x}' in X_temp.columns]
                for col in m_cols:
                    # For M features, which could be categorical, first convert to numeric if needed
                    if not pd.api.types.is_numeric_dtype(X_temp[col]):
                        # Use factorize to convert categorical to numeric codes
                        X_temp[f'{col}_fact'] = pd.factorize(X_temp[col])[0]
                        num_col = f'{col}_fact'
                    else:
                        num_col = col
                    
                    agg_col = f'{col}_mean_by_uid'
                    agg_result = X_temp.groupby('uid')[num_col].mean().reset_index()
                    agg_result.columns = ['uid', agg_col]
                    self.agg_stats[agg_col] = agg_result
                    agg_features.append(agg_col)
                
                # Other specific features
                other_cols = ['P_emaildomain', 'dist1', 'id_02']
                if 'dist1' not in X_temp.columns and 'dist2' in X_temp.columns:
                    X_temp['dist1'] = X_temp['dist2']  # Use dist2 as a fallback
                
                for col in other_cols:
                    if col in X_temp.columns:
                        agg_col = f'{col}_mean_by_uid'
                        # For categorical features like P_emaildomain, use factorize
                        if not pd.api.types.is_numeric_dtype(X_temp[col]):
                            X_temp[f'{col}_fact'] = pd.factorize(X_temp[col])[0]
                            agg_result = X_temp.groupby('uid')[f'{col}_fact'].mean().reset_index()
                        else:
                            agg_result = X_temp.groupby('uid')[col].mean().reset_index()
                        agg_result.columns = ['uid', agg_col]
                        self.agg_stats[agg_col] = agg_result
                        agg_features.append(agg_col)
                
                # Special V features
                v_special_cols = ['V127', 'V136', 'V309', 'V307', 'V320', 'V314']
                v_special_cols = [col for col in v_special_cols if col in X_temp.columns]
                for col in v_special_cols:
                    # Make sure the column is numeric before aggregating
                    if pd.api.types.is_numeric_dtype(X_temp[col]):
                        agg_col = f'{col}_mean_by_uid'
                        agg_result = X_temp.groupby('uid')[col].mean().reset_index()
                        agg_result.columns = ['uid', agg_col]
                        self.agg_stats[agg_col] = agg_result
                        agg_features.append(agg_col)
                
                # Add special std aggregation for C14
                if 'C14' in X_temp.columns and pd.api.types.is_numeric_dtype(X_temp['C14']):
                    agg_col = 'C14_std_by_uid'
                    agg_result = X_temp.groupby('uid')['C14'].std().reset_index()
                    agg_result.columns = ['uid', agg_col]
                    self.agg_stats[agg_col] = agg_result
                    agg_features.append(agg_col)
        
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        
        # Create features related to missing values
        X_transformed['missing_count'] = X_transformed.isna().sum(axis=1)
        X_transformed['missing_percentage'] = X_transformed['missing_count'] / len(X_transformed.columns)
        
        # Create aggregate features from Vesta features
        if self.vesta_cols:
            vesta_columns = [col for col in self.vesta_cols if col in X_transformed.columns]
            if vesta_columns:
                # Filter to only numeric columns to avoid errors
                numeric_vesta = [col for col in vesta_columns if pd.api.types.is_numeric_dtype(X_transformed[col])]
                if numeric_vesta:
                    X_transformed['vesta_mean'] = X_transformed[numeric_vesta].mean(axis=1)
                    X_transformed['vesta_std'] = X_transformed[numeric_vesta].std(axis=1)
            
        # Create features from amount and time variables
        if 'TransactionAmt' in X_transformed.columns and pd.api.types.is_numeric_dtype(X_transformed['TransactionAmt']):
            # Log transform for transaction amount
            X_transformed['TransactionAmt_Log'] = np.log1p(X_transformed['TransactionAmt'])
            
            # Amount fractional part - fraudsters may use specific patterns
            X_transformed['TransactionAmt_Decimal'] = X_transformed['TransactionAmt'] - np.floor(X_transformed['TransactionAmt'])
            
            # Round amount indicators
            X_transformed['TransactionAmt_IsRound'] = (X_transformed['TransactionAmt_Decimal'] == 0).astype(int)
            
            # Add cents feature
            X_transformed['cents'] = (X_transformed['TransactionAmt'] - np.floor(X_transformed['TransactionAmt'])) * 100
            
        # Create features from card information
        if 'card4' in X_transformed.columns and 'card6' in X_transformed.columns:
            # Card type and transaction type combination
            X_transformed['card4_card6'] = X_transformed['card4'].astype(str) + "_" + X_transformed['card6'].astype(str)
            
        # Calculate day and UID features for aggregation
        if 'TransactionDT' in X_transformed.columns and pd.api.types.is_numeric_dtype(X_transformed['TransactionDT']):
            # Calculate day from TransactionDT
            X_transformed['day'] = X_transformed['TransactionDT'] / (24*60*60)
            
            # Extract hour of the day (0-23)
            X_transformed['Hour'] = ((X_transformed['TransactionDT'] / 3600) % 24).astype(int)
            
            # Weekend indicator (assuming day starts at 0)
            X_transformed['Weekend'] = (((X_transformed['TransactionDT'] / 86400) % 7) >= 5).astype(int)
            
            # Create month difference feature (DT_M)
            # DT_M (difference in months)
            X_transformed['DT_M'] = (X_transformed['TransactionDT'] / (24*60*60*30)).astype(int)
        
        # Create UID feature if necessary columns exist
        if all(col in X_transformed.columns for col in ['card1', 'addr1', 'day', 'D1']):
            # Create card1_addr1 combination
            X_transformed['card1_addr1'] = X_transformed['card1'].astype(str) + '_' + X_transformed['addr1'].astype(str)
            
            # Create UID feature
            X_transformed['uid'] = X_transformed['card1_addr1'].astype(str) + '_' + (np.floor(X_transformed['day'] - X_transformed['D1'])).astype(str)
            
            # Apply stored aggregation statistics
            for agg_col, agg_df in self.agg_stats.items():
                X_transformed = X_transformed.merge(agg_df, on='uid', how='left')
        
        # Create outsider15 feature
        if 'D1' in X_transformed.columns and 'D15' in X_transformed.columns and \
           pd.api.types.is_numeric_dtype(X_transformed['D1']) and pd.api.types.is_numeric_dtype(X_transformed['D15']):
            X_transformed['outsider15'] = (np.abs(X_transformed['D1'] - X_transformed['D15']) > 3).astype(int)
        
        # Email domain features if available
        if 'P_emaildomain' in X_transformed.columns:
            X_transformed['P_emaildomain_category'] = X_transformed['P_emaildomain'].apply(
                lambda x: str(x).split('.')[0] if isinstance(x, str) else 'unknown')
        
        return X_transformed
  

def build_preprocessing_pipeline(feature_lists):
    """
    Build the preprocessing pipeline for different feature types
    
    Parameters:
    - feature_lists: Dictionary containing lists of different feature types
    
    Returns:
    - ColumnTransformer for preprocessing
    """
    with mlflow.start_run(run_name="preprocessing_pipeline_creation"):
        # Numeric pipeline: impute and scale
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        # Categorical pipeline: impute and encode
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        # Combine numeric and categorical transformers
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, feature_lists['numeric_features']),
                ('cat', categorical_transformer, feature_lists['categorical_features'])
            ])
        
        # Log preprocessing info
        mlflow.log_param("num_numeric_features", len(feature_lists['numeric_features']))
        mlflow.log_param("num_categorical_features", len(feature_lists['categorical_features']))
        
        return preprocessor

def build_full_pipeline_enhanced(preprocessor):
    """
    Build the full pipeline including enhanced feature engineering, preprocessing, and model
    
    Parameters:
    - preprocessor: ColumnTransformer for preprocessing
    
    Returns:
    - Complete sklearn Pipeline
    """
    with mlflow.start_run(run_name="full_pipeline_creation"):
        # Feature engineering first, then preprocessing, then model
        pipeline = Pipeline(steps=[
            ('feature_engineering', EnhancedFeatureEngineering()),
            ('preprocessor', preprocessor),
            ('classifier', DecisionTreeClassifier(
                max_depth=8,
                min_samples_split=100,
                min_samples_leaf=50,
                class_weight='balanced',
                random_state=42))
        ])
        
        # Log pipeline parameters
        mlflow.log_param("max_depth", 8)
        mlflow.log_param("min_samples_split", 100)
        mlflow.log_param("min_samples_leaf", 50)
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("magic_features_enabled", True)
        
        # Save pipeline description
        with open("pipeline_description.txt", "w") as f:
            f.write("Full Pipeline with Magic Features:\n")
            f.write("1. Enhanced Feature Engineering - Including UID Magic Feature\n")
            f.write("2. Preprocessing - Column transformer with numeric and categorical pipelines\n")
            f.write("3. Decision Tree Classifier\n")
        
        mlflow.log_artifact("pipeline_description.txt")
    
    return pipeline

def train_evaluate_model(pipeline, X_train, X_test, y_train, y_test):
    """
    Train the model pipeline and evaluate its performance
    
    Parameters:
    - pipeline: Full sklearn Pipeline
    - X_train: Training features
    - X_test: Testing features
    - y_train: Training labels
    - y_test: Testing labels
    
    Returns:
    - Trained pipeline
    """
    with mlflow.start_run(run_name="model_training_evaluation"):
        # Log dataset sizes
        mlflow.log_param("train_size", X_train.shape[0])
        mlflow.log_param("test_size", X_test.shape[0])
        
        # Fit the pipeline on training data
        print("Training model...")
        pipeline.fit(X_train, y_train)
        
        # Make predictions on test data
        print("Generating predictions...")
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
        y_pred = pipeline.predict(X_test)
        
        # Calculate metrics
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
        pr_auc = auc(recall, precision)
        
        # Log metrics
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("pr_auc", pr_auc)
        
        # Generate and log confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix')
        plt.savefig('confusion_matrix.png')
        mlflow.log_artifact('confusion_matrix.png')
        plt.close()
        
        # Generate and log ROC curve
        plt.figure(figsize=(8, 6))
        fpr, tpr, _ = precision_recall_curve(y_test, y_pred_proba)
        plt.plot(fpr, tpr, marker='.')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curve')
        plt.savefig('pr_curve.png')
        mlflow.log_artifact('pr_curve.png')
        plt.close()
        
        # Log classification report
        report = classification_report(y_test, y_pred)
        with open("classification_report.txt", "w") as f:
            f.write(report)
        mlflow.log_artifact("classification_report.txt")
        
        print(f"Model evaluation complete. ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}")
        
        return pipeline

def main_enhanced():
    """Main function to run the entire pipeline with enhanced feature engineering"""
    print("Starting IEEE-CIS Fraud Detection Pipeline with Magic Features")
    
    # Define data paths - update these to your actual file paths
    # transaction_path = "train_transaction.csv"  # Update this path
    # identity_path = "train_identity.csv"  # Update this path
    
    # Step 1: Load data
    print("Loading data...")
    df = load_ieee_cis_data(transaction_path, identity_path)
    
    # Step 2: Analyze and clean data
    print("Cleaning data...")
    # Use the fixed version of analyze_and_clean_data
    columns = list(df.columns)
    dtypes = [str(df[col].dtype) for col in columns]
    non_null_counts = [df[col].count() for col in columns]
    nunique_values = [df[col].nunique() for col in columns]
    memory_usage = [df[col].memory_usage(deep=True) / 1024**2 for col in columns]
    
    # Create DataFrame with lists of equal length for data quality check
    data_quality = pd.DataFrame({
        'column': columns,
        'dtype': dtypes,
        'non_null_count': non_null_counts,
        'nunique': nunique_values,
        'memory_usage': memory_usage
    })
    
    # Now proceed with the cleaning
    cleaned_df = analyze_and_clean_data(df)
    
    # Step 3: Prepare feature lists
    print("Identifying features...")
    feature_lists = prepare_feature_lists(cleaned_df)
    
    # Step 4: Prepare train/test split
    print("Splitting data...")
    X = cleaned_df.drop('isFraud', axis=1)
    y = cleaned_df['isFraud']
    
    # Split with stratification to handle class imbalance
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Step 5: Build preprocessing pipeline
    print("Building preprocessing pipeline...")
    preprocessor = build_preprocessing_pipeline(feature_lists)
    
    # Step 6: Build full pipeline with enhanced feature engineering
    print("Building full pipeline with Magic Features...")
    full_pipeline = build_full_pipeline_enhanced(preprocessor)
    
    # Step 7: Train and evaluate model
    print("Training and evaluating model...")
    trained_pipeline = train_evaluate_model(full_pipeline, X_train, X_test, y_train, y_test)
    
    print("Pipeline complete! All runs and metrics have been logged to MLflow.")
    print("Magic Features have been added to improve fraud detection.")
    print("To view results, run 'mlflow ui' in your terminal.")

def predict_with_model(pipeline, transaction_path, identity_path=None, output_path="predictions.csv"):
    """
    Generate predictions using a trained pipeline
    
    Parameters:
    - pipeline: Trained sklearn Pipeline
    - transaction_path: Path to transaction data
    - identity_path: Optional path to identity data
    - output_path: Path to save predictions
    
    Returns:
    - DataFrame with predictions
    """
    print("Loading test data...")
    test_df = load_ieee_cis_data(transaction_path, identity_path)
    
    # Save TransactionID for output
    transaction_ids = test_df['TransactionID'].copy()
    
    print("Generating predictions...")
    # Generate fraud probability scores
    if hasattr(pipeline, 'predict_proba'):
        predictions = pipeline.predict_proba(test_df)[:, 1]
    else:
        predictions = pipeline.predict(test_df)
    
    # Create output DataFrame
    output_df = pd.DataFrame({
        'TransactionID': transaction_ids,
        'isFraud': predictions
    })
    
    # Save predictions
    output_df.to_csv(output_path, index=False)
    print(f"Predictions saved to {output_path}")
    
    return output_df

In [ ]:

main_enhanced()

Starting IEEE-CIS Fraud Detection Pipeline with Magic Features
Loading data...
Transaction data loaded: (590540, 394)
Identity data loaded: (144233, 41)
Merged data shape: (590540, 434)
Cleaning data...
🏃 View run data_cleaning at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6/runs/08d8023883bd4b2994329cab5161ab6d
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6
Identifying features...
🏃 View run feature_identification at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6/runs/aacb91a0a5f74d8597d000a8b81c6a76
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6
Splitting data...
Building preprocessing pipeline...
🏃 View run preprocessing_pipeline_creation at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6/runs/90272a69dbb94b45b68e2acf3559f250
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/6
Building full pipeline w

In [6]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# --- Custom Transformers ---

class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.start_date = pd.to_datetime('2017-12-01')
        self.label_encoders = {}

    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns.tolist()
        categorical_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in categorical_cols:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            self.label_encoders[col] = le
        return self

    def transform(self, X):

        if 'TransactionDT' in X.columns:
            X['Transaction_hour'] = ((X['TransactionDT']// (60*60)) % 24).astype('int8')
            X['Transaction_day'] = ((X['TransactionDT']// (24*60*60)).astype('int16'))
            X['Transaction_weekday'] = X['Transaction_day'] % 7
            # X['Transaction_month'] = X['TransactionDT'].dt.month
            # X['Transaction_year'] = X['TransactionDT'].dt.year

        for col, le in self.label_encoders.items():
            X[col] = le.transform(X[col].astype(str))

        return X

class NumericalFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, numerical_columns=None):
        self.numerical_columns = numerical_columns

    def fit(self, X, y=None):
        if self.numerical_columns is None:
            self.numerical_columns = X.select_dtypes(include=[np.number]).columns.tolist()
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=self.numerical_columns)
        return X[self.numerical_columns].copy()

class NanPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.fill_values = {}

    def fit(self, X, y=None):
        for col in X.columns:
            if self.strategy == 'mean':
                self.fill_values[col] = X[col].mean()
            elif self.strategy == 'median':
                self.fill_values[col] = X[col].median()
            elif self.strategy == 'mode':
                self.fill_values[col] = X[col].mode()[0]
            else:
                raise ValueError(f"Unknown strategy: {self.strategy}")
        return self

    def transform(self, X):
        X = X.copy()
        for col, fill_value in self.fill_values.items():
            X[col] = X[col].fillna(fill_value)
        return X

class UserIDCreator(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.version == 1:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str) + '_' + \
                           X['Transaction_hour'].astype(str)
        else:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['P_emaildomain'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str)
        return X

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.columns:
            self.freq_maps[col] = X[col].value_counts()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[f'{col}_freq_enc'] = X[col].map(self.freq_maps[col])
        return X

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, target_col):
        self.columns = columns
        self.target_col = target_col
        self.woe_maps = {}

    def fit(self, X, y=None):
        eps = 1e-6
        for col in self.columns:
            temp = X.groupby(col)[self.target_col].agg(['sum', 'count'])
            temp['good'] = temp['count'] - temp['sum']
            temp['bad_rate'] = (temp['sum'] + eps) / (temp['count'] + eps)
            temp['good_rate'] = (temp['good'] + eps) / (temp['count'] + eps)
            temp['woe'] = np.log(temp['good_rate'] / temp['bad_rate'])
            self.woe_maps[col] = temp['woe'].to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[f'{col}_woe_enc'] = X[col].map(self.woe_maps[col])
        return X

class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, target_col):
        self.columns = columns
        self.target_col = target_col
        self.target_maps = {}

    def fit(self, X, y=None):
        for col in self.columns:
            self.target_maps[col] = X.groupby(col)[self.target_col].mean()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[f'{col}_target_enc'] = X[col].map(self.target_maps[col])
        return X

# --- Pipeline Creator ---

def FeaturePipeline():
    pipeline = Pipeline([
        ('time_features', TimeFeatureExtractor()),
        ('nan_preprocessing', NanPreprocessor(strategy='mean')),  # 🤓 Smart NaN fix
        ('numerical_features', NumericalFeatureExtractor()),
        ('user_id', UserIDCreator(version=1)),
        ('freq_encoding', FrequencyEncoder(columns=['card1', 'P_emaildomain'])),
        ('woe_encoding', WOEEncoder(columns=['card1', 'P_emaildomain'], target_col='isFraud')),
        ('target_encoding', TargetEncoder(columns=['card1', 'P_emaildomain'], target_col='isFraud')),
    ])
    return pipeline


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
    
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def custom_user_time_split(X, y, user_col='user_id', time_col='TransactionDT', test_size=0.2, random_state=42):
    # 1. Create a copy to avoid messing up
    X_ = X.copy()
    y_ = y.copy()

    # 2. Group by user_id: get min timestamp per user
    user_min_time = X_.groupby(user_col)[time_col].min()

    # 3. Sort users by their first transaction
    user_min_time = user_min_time.sort_values()

    # 4. Create train/test user splits based on first transaction time
    n_users = len(user_min_time)
    n_test_users = int(test_size * n_users)

    np.random.seed(random_state)
    test_users = user_min_time.index[-n_test_users:]  # latest users for test
    train_users = user_min_time.index[:-n_test_users]

    # 5. Split X and y based on user_id
    X_train = X_[X_[user_col].isin(train_users)].reset_index(drop=True)
    X_test = X_[X_[user_col].isin(test_users)].reset_index(drop=True)
    y_train = y_[X_[user_col].isin(train_users)].reset_index(drop=True)
    y_test = y_[X_[user_col].isin(test_users)].reset_index(drop=True)

    print(f"✅ Split done! Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    return X_train, X_test, y_train, y_test



In [8]:
X = df.copy()
y = X.pop('isFraud')

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, 0.2, )

# 2. Feature Pipeline
feature_pipeline = FeaturePipeline()
feature_pipeline.fit(X_train)

custom_user_time_split(X, y, user_col='user_id', time_col='TransactionDT')

X_train_proc = feature_pipeline.transform(X_train)
X_test_proc = feature_pipeline.transform(X_test)

# 3. Log processed dataset
log_preprocessed_data(X_train_proc, step_name="Feature Engineering")
log_preprocessed_data(X_test_proc, step_name="Feature Engineering")

# 4. Train model
from sklearn.tree import DecisionTreeClassifier  # or Regressor, depending on task
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_proc, y_train)

# 5. Log model
log_model(model, step_name="DecisionTree_Training")

print("🚀 All done! Check your experiment logs!")

KeyError: 'user_id'

In [28]:
X_train.shape

(590540, 404)

In [29]:
X_test.shape

(0, 404)

In [ ]:
# 5. Train Decision Tree
model = DecisionTreeClassifier(max_depth=6, random_state=42)
print(print(X_train.dtypes))
from sklearn.preprocessing import LabelEncoder

In [ ]:
model.fit(X_train, y_train)

# 6. Predict
preds = model.predict(X_test)

In [ ]:
# 7. Evaluation
print("\n🎯 Accuracy:", accuracy_score(y_test, preds))
print("\n📜 Classification Report:\n", classification_report(y_test, preds))

# Optional: plot feature importance
feat_importances = pd.Series(model.feature_importances_, index=features)
feat_importances.nlargest(15).plot(kind='barh')
plt.title('Top Feature Importances')
plt.show()

return model